In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

import pandas as pd

df = pd.read_csv("/kaggle/input/q1-ka-ai-2026/Q1_data.csv")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.hist(df['Delivery_Time'], bins=20, edgecolor='black')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.title('Distribution of Delivery Time')
plt.show()



In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])


In [ ]:
# Task 2: Write your code here:
df = df.fillna(df.mode().iloc[0])
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop='first', sparse_output=False)

df = pd.concat(
    [df.drop(columns=['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']),
     pd.DataFrame(
         encoder.fit_transform(df[['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']]),
         columns=encoder.get_feature_names_out(['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']),
         index=df.index
     )],
    axis=1
)


In [ ]:
# see all columns
print(df.columns)

# see first rows
df.head()

# shape before / after
df.shape


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 6: Write your code here:

In [ ]:

# Task 1: Split the dataset into features (X) and target (y)
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Task 2: Use KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

# Tasks 3 & 4: Train RandomForest and evaluate using MAE
for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

# Task 5: Print average MAE across all folds
print("Average MAE across all folds:", np.mean(mae_scores))
print("Average Accuracy:", np.mean(scores) * 100, "%")




In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Train model on full dataset
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X, y)

importances = rf_model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.gca().invert_yaxis()
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance')
plt.show()


In [ ]:
# Task 2: Write your code here:
y_pred_full = rf_model.predict(X)

plt.figure(figsize=(6,4))
plt.hist(y_pred_full, bins=20, edgecolor='black')
plt.xlabel('Predicted Delivery Time')
plt.ylabel('Frequency')
plt.title('Predicted Delivery Time Distribution')
plt.show()


In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:

from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf = RandomForestRegressor(random_state=42)
    cb = CatBoostRegressor(verbose=0, random_state=42)

    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    preds = (rf.predict(X_val) + cb.predict(X_val)) / 2
    mae_scores.append(mean_absolute_error(y_val, preds))

# Results
avg_mae = np.mean(mae_scores)
mean_delivery_time = y.mean()
accuracy_percent = (1 - avg_mae / mean_delivery_time) * 100

print("Ensemble MAE:", avg_mae)
print("Ensemble Accuracy (%):", round(accuracy_percent, 2))